In [1]:
from pathlib import Path
import pandas as pd
import sys

In [6]:
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
DATA_PATH


PosixPath('/Users/srijanshovit/Documents/Coding/SignatureVerificationSiamese/data')

In [7]:
def analyze_signature_dataset(root_dir: str):
    root = Path(root_dir)
    records = []

    for split in ["train", "test"]:
        split_path = root / split
        if not split_path.exists():
            continue

        for folder in filter(Path.is_dir, split_path.iterdir()):
            folder_name = folder.name
            person_id = folder_name.replace("_forg", "") if folder_name.endswith("_forg") else folder_name
            signature_type = "forged" if folder_name.endswith("_forg") else "real"
            num_images = len(list(folder.iterdir()))

            records.append({
                "split": split,
                "person_id": person_id,
                "type": signature_type,
                "num_images": num_images
            })

    return pd.DataFrame(records)

### Raw Folder Stats

In [8]:
df = analyze_signature_dataset(DATA_PATH)

print(f"Total folders:{len(df)}")
df.sample(10)



Total folders:170


,split,person_id,type,num_images
59,train,069,forged,12
85,train,026,real,12
129,test,057,forged,12
159,test,068,real,12
105,train,034,real,12
101,train,022,forged,16
45,train,030,real,12
161,test,056,real,12
111,train,056,real,12
97,train,066,real,12


### Aggregated Counts by Split and Type

In [9]:
summary = df.groupby(["split", "type"])["num_images"].sum()
summary

split  type  
test   forged    248
       real      252
train  forged    762
       real      887
Name: num_images, dtype: int64

### Per Person Summary

In [10]:
per_person = df.groupby(
    ["split", "person_id", "type"]
)["num_images"].sum().unstack(fill_value=0)

per_person.head(10)

type             forged  real
split person_id              
test  049            12    12
      050            12    12
      051             8    12
      052            16    12
      053            16    12
      054            20    12
      055            12    12
      056             8    12
      057            12    12
      058            16    12